# Analyze Energy Ratios during Baseline Operation

In this notebook, we will demonstrate how to compute and plot the energy ratio between test and reference turbines as a function of wind direction. We'll focus on baseline operation for this example (i.e., without wake steering). The energy ratios can be used to evaluate wake losses experienced by different turbines.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from flasc.data_processing import dataframe_manipulations as dfm
from flasc.utilities import floris_tools as ftools
from flasc.utilities.utilities_examples import load_floris_smarteole as load_floris

In [2]:
# Suppress warnings
import warnings

warnings.filterwarnings("ignore")

# Step 0: Load processed data

Load the processed SCADA data with power curve filtering and northing calibration applied and inspect

In [ ]:
def load_data():
    root_path = Path.cwd()
    f = root_path / "postprocessed" / "df_scada_data_60s_filtered_and_northing_calibrated.pkl"
    df_scada = pd.read_pickle(f)
    return df_scada


df_scada = load_data()

# Step 1. Prepare FLORIS object

description...

In [4]:
# Load FLORIS model of site
fm, turbine_weights = load_floris()

In [ ]:
# Use FLORIS to identify upstream / unwaked turbines for
# each direction
df_upstream = ftools.get_upstream_turbs_floris(fm)
df_upstream.head()

,wd_min,wd_max,turbines
0,0.0,25.0,[0]
1,25.0,28.0,"[0, 2]"
2,28.0,31.0,"[0, 2, 6]"
3,31.0,31.3,"[0, 2, 3, 6]"
4,31.3,31.5,"[0, 2, 3, 4, 6]"


In [6]:
# Use flasc tools to establish reference wind speeds and directions

# Since will be interested in looking at impacts on SMV5/[4], exclude
# it from each calculation

# Set the wind direction as the average of all turbine averages
df_scada = dfm.set_wd_by_turbines(df_scada, [0, 1, 2, 3, 5, 6])

# Set the wind speed to be the average of all upstream turbines
# (turbines not in a wake in a given direction)
# Except for SMV5
df_scada = dfm.set_ws_by_upstream_turbines(df_scada, df_upstream, exclude_turbs=[4])

# Set the reference power to the average of all upstream turbines
# Except for SMV5
df_scada = dfm.set_pow_ref_by_upstream_turbines(df_scada, df_upstream, exclude_turbs=[4])

# Step 2: Generate a fake LES timeseries based on FLORIS solutions and noise

Description...

In [20]:
# Load a FLORIS precalculated solutions table
floris_path = Path.cwd() / "precalculated_floris_solutions"
wake_model = "turbopark"
fn = floris_path / "df_fi_approx_{:s}.ftr".format(wake_model)
if fn.is_file():
    df_fm_approx = pd.read_feather(fn)
else:
    raise UserWarning("Requires a precalculated FLORIS solutions file.")

In [ ]:
# Now generate a 10-min-averaged timeseries for LES with inflow similar to the SCADA
from flasc.data_processing import time_operations as tops
from floris.utilities import wrap_360

new_time_array = pd.date_range(start=df_scada.iloc[0]["time"], end=df_scada.iloc[-1]["time"], freq="60s")
df_res = tops.df_resample_by_interpolation(
    df=df_scada[["time", "wd", "ws", "ti"]],
    time_array=new_time_array,
    circular_cols=["wd"],
    interp_method="linear",
    verbose=True,
    max_gap=np.timedelta64(9999999, 's')
)
df_les = ftools.interpolate_floris_from_df_approx(
    df=df_res,
    df_approx=df_fm_approx,
    method="linear",
    verbose=True,
    mirror_nans=False,
    extrapolate_ws=True
)
df_les = df_les.rename(columns={"wd": "wd_metmast", "ws": "ws_metmast", "ti": "ti_metmast"})


2025-11-04 13:48:56   Resampling column 'wd' with median timestep 60.000 s onto a prespecified time array with kind=linear, max_gap=9999999.0s, and wrap_around_360=True
2025-11-04 13:48:56   Resampling column 'ws' with median timestep 60.000 s onto a prespecified time array with kind=linear, max_gap=9999999.0s, and wrap_around_360=False
2025-11-04 13:48:56   Resampling column 'ti' with median timestep 60.000 s onto a prespecified time array with kind=linear, max_gap=9999999.0s, and wrap_around_360=False
2025-11-04 13:48:56 Warning: not mirroring NaNs from the raw data to the FLORIS predictions. This may skew your energy ratios.
2025-11-04 13:48:56 Identified the following grid type: 3d.
2025-11-04 13:48:56 Warning: the values in df[ws] exceed the range in the precalculated solutions df_fi_approx[ws].
2025-11-04 13:48:56    minimum/maximum value in df:        (0.250, 18.865)
2025-11-04 13:48:56    minimum/maximum value in df:        (0.250, 18.865)
2025-11-04 13:48:56    minimum/maximum

In [ ]:
# Now add noise to the mesoscale inflow
df_les["ti_metmast"] = 0.0633 + 0.10 * np.random.rand() * df_les["ti_metmast"]
df_les["wd_metmast"] = wrap_360(df_les["wd_metmast"] + 0.5 * np.random.randn() + np.random.randn(df_les.shape[0]))  # bias plus Normal noise
df_les["ws_metmast"]+= 0.5 * np.random.randn() + np.random.randn(df_les.shape[0])  # bias plus Normal noise

# And finally save this as a 'fake' LES timeseries
print(df_les)
df_les = df_les[["time", "wd_metmast", "ws_metmast", "ti_metmast", "pow_000", "pow_001", "pow_002", "pow_003", "pow_004", "pow_005", "pow_006"]]
df_les.to_csv("./data/SMARTEOLE-LES-simulation-data/les_timeseries.csv", index=False)

FlascDataFrame in FLASC format
       wd_metmast  ws_metmast  ti_metmast                time       pow_000  \
0      251.489594   12.171596    0.072102 2020-02-17 16:30:00  1.957574e+06   
1      252.499648   12.629076    0.072102 2020-02-17 16:40:00  1.991138e+06   
2      256.239965   10.795476    0.072102 2020-02-17 16:50:00  1.852258e+06   
3      255.154876   10.620064    0.072102 2020-02-17 17:00:00  1.825858e+06   
4      251.485556   10.690601    0.072102 2020-02-17 17:10:00  1.714986e+06   
...           ...         ...         ...                 ...           ...   
14008  307.803805    5.428880    0.072102 2020-05-24 23:10:00  1.480561e+05   
14009  305.823873    2.799786    0.072102 2020-05-24 23:20:00  1.830347e+05   
14010  304.555667    4.501962    0.072102 2020-05-24 23:30:00  1.477709e+05   
14011  300.246204    2.970565    0.072102 2020-05-24 23:40:00  1.526191e+05   
14012  305.211153    5.069239    0.072102 2020-05-24 23:50:00  1.970683e+05   

            pow_001 